# 04 · Middleware and a dynamic agent
**The limitation this fixes:** every user gets the same prompt and every tool, and our safety rules are scattered around.

| Need | Framework piece |
|---|---|
| A tool that changes agent state | tool returns `Command(update=...)` |
| Our own state fields | `state_schema` |
| Tools by role and identity, model by role | `@wrap_model_call` |
| Prompt by role and language | `@dynamic_prompt` |
| Long chats, outages, cost, flaky tools | `SummarizationMiddleware`, `ModelFallbackMiddleware`, `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `ToolRetryMiddleware` |

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far

In [ ]:
from common import (CareContext, RAG_PROMPT, PATIENTS, FALLBACK_MODEL,
                    get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse,
                    build_leaflet_search, dose_guard, pii_middleware, audit_log,
                    list_free_slots, book_appointment, patients_needing_attention,   # plain tools, like in 01
                    run_eval, ask, resume, show_state)
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

search_documents = build_leaflet_search()

## 1. A tool that updates state: `verify_identity`
Tools usually return text. Returning a `Command` lets a tool also write to the agent's **state**.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def verify_identity(date_of_birth: str, runtime: ToolRuntime[CareContext]) -> Command:
    """Verify the patient's identity with their date of birth (YYYY-MM-DD). Required before booking."""
    ok = PATIENTS[runtime.context.patient_token]["date_of_birth"] == date_of_birth
    reply = "Identity verified." if ok else "That date of birth doesn't match our records."
    return Command(update={"identity_verified": ok,
                           "messages": [ToolMessage(reply, tool_call_id=runtime.tool_call_id)]})

## 2. Add our own field to the agent's state

In [ ]:
from typing import NotRequired
from langchain.agents import AgentState

class CareState(AgentState):          # messages + our flag
    identity_verified: NotRequired[bool]

## 3. `@wrap_model_call`: rebuild the tool list (and model) on every call
Patients never see doctor tools. Booking tools only appear after identity is verified. Doctors get a stronger model.

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

SHOW_TOOLS = True                                          # workshop receipt; turned off for the benchmark
DOCTOR_ONLY = {"patients_needing_attention"}               # CUSTOM: access policy
NEEDS_IDENTITY = {"list_free_slots", "book_appointment"}   # CUSTOM: access policy
strong_model = init_chat_model(FALLBACK_MODEL)

@wrap_model_call(state_schema=CareState)
def pick_tools_and_model(request, handler):
    role = request.runtime.context.role
    tools = request.tools
    if role != "doctor":
        tools = [t for t in tools if t.name not in DOCTOR_ONLY]
    if not request.state.get("identity_verified"):
        tools = [t for t in tools if t.name not in NEEDS_IDENTITY]
    if SHOW_TOOLS:
        print("   🔧 tools this call:", [t.name for t in tools])
    model = strong_model if role == "doctor" else request.model
    return handler(request.override(tools=tools, model=model))

## 4. `@dynamic_prompt`: the system prompt is written fresh on every call

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def care_prompt(request):
    ctx = request.runtime.context
    tone = ("You are talking to a doctor. Be brief and clinical."
            if ctx.role == "doctor" else "You are talking to a patient. Use plain, warm language.")
    return (f"{RAG_PROMPT}\n- Before booking, ask for date of birth and call verify_identity.\n"
            f"{tone}\nAlways reply in {ctx.language}.")

## 5. The whole stack in one list
Almost every line is a LangChain import. **Order matters:** it runs top to bottom on the way in, like layers of an onion.
Redaction sits above summarization, so identifiers never reach a summary.

In [ ]:
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware, ModelCallLimitMiddleware, ModelFallbackMiddleware,
    SummarizationMiddleware, ToolCallLimitMiddleware, ToolRetryMiddleware)

middleware = [
    dose_guard,                                                   # 02: refuse dose questions
    *pii_middleware(),                                            # 03: redact MRNs and phones
    SummarizationMiddleware(MODEL, trigger=("messages", 40), keep=("messages", 20)),   # long chats
    care_prompt,                                                  # prompt by role and language
    pick_tools_and_model,                                         # tools by role and identity
    ModelFallbackMiddleware(FALLBACK_MODEL),                      # model down? try another
    ModelCallLimitMiddleware(run_limit=10),                       # cost guard
    ToolCallLimitMiddleware(run_limit=10),                        # cost guard
    ToolRetryMiddleware(max_retries=2),                           # flaky tools
    audit_log,                                                    # 03: who touched what
    HumanInTheLoopMiddleware(interrupt_on={"escalate_to_nurse": True}),   # 02: nurse approves
]

tools = [get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse, search_documents,
         verify_identity, list_free_slots, book_appointment, patients_needing_attention]
agent = create_agent(MODEL, tools=tools, context_schema=CareContext,
                     checkpointer=InMemorySaver(), middleware=middleware)

## Demo: booking tools appear only after verification
Watch the 🔧 line grow.

In [ ]:
ask(agent, "I'd like to book an appointment with my doctor", thread="book")
ask(agent, "1970-03-14", thread="book")

In [ ]:
ask(agent, "The first slot please", thread="book")
agent.get_state({"configurable": {"thread_id": "book"}}).values["identity_verified"]   # 🧾 receipt: our state field

## ❌ Failure case: wrong date of birth, no booking tools

In [ ]:
ask(agent, "Book me an appointment. My date of birth is 1999-01-01", thread="impostor")

## Same agent, doctor context: different tools, model and tone

In [ ]:
dr_lee = CareContext(user_id="dr_lee", role="doctor")
ask(agent, "Who needs attention today?", thread="doctor", context=dr_lee)
ask(agent, "Who needs attention today?", thread="patient-tries")   # patient: tool isn't even offered

In [ ]:
ask(agent, "What's my blood pressure target?", thread="es", context=CareContext(language="Spanish"))

## Rerun the benchmark
Same dataset, new experiment. Compare it against `02-baseline` in LangSmith.

In [ ]:
SHOW_TOOLS = False   # quiet: the benchmark makes ~20 model calls
results = run_eval(agent, "04-middleware")
results.to_pandas()[["inputs.question", "feedback.used_right_tool",
                     "feedback.mentions_key_fact", "feedback.paused_when_expected"]]

## Where this lives in the backend
- `app/middleware.py: the one `middleware` list, `pick_tools_and_model`, `care_prompt``
- `app/tools.py: `verify_identity`, booking tools`

Same primitives, just split into files. The notebook never imports `app/`.